# DeepCT (2019)
---
[[paper]](https://arxiv.org/pdf/1910.10683)
<br>DeepCT = Deep Contextualized Term Weighting

DeepCT — это подход к **контекстуализированному взвешиванию терминов** в документах, который использует предварительно обученные языковые модели (такие как BERT) для создания более семантически насыщенных представлений документов для систем информационного поиска. Он был предложен исследователями из Microsoft Research.

### Контекст
Традиционные методы информационного поиска, такие как TF-IDF (1972) и BM25 (1994), основываются на статистической частоте терминов. Они эффективно работают на уровне ключевых слов, но часто не улавливают семантические нюансы и контекст, что приводит к упущению релевантных документов, использующих синонимы или схожие по смыслу фразы. В то же время, нейронные модели, такие как BERT (2018), способны глубоко понимать контекст и семантику, но их применение для индексирования больших корпусов документов для первичного (first-stage) поиска затруднено из-за высокой вычислительной стоимости инференса для каждой пары запрос-документ.

### Идея метода
Основная идея DeepCT заключается в том, чтобы **использовать возможности контекстуального понимания нейронных языковых моделей (LLM) для обогащения традиционных инвертированных индексов**, не отказываясь от их скорости и масштабируемости. Вместо того чтобы просто считать частоту слова, DeepCT оценивает *важность* каждого слова в документе, исходя из его контекста в этом документе. Эти контекстуальные веса затем используются для индексации, делая классический sparse retrieval более семантически осведомленным.

### Постановка задачи
DeepCT решает задачу **улучшения представления документов для фазы первого прохода (first-stage retrieval)** в системах информационного поиска. Цель состоит в том, чтобы обеспечить более точное ранжирование документов по релевантности к запросу, особенно в случаях, когда прямые совпадения ключевых слов недостаточны, за счет более интеллектуального взвешивания терминов внутри документов.

### Существующие альтернативы
На момент появления DeepCT существовали следующие основные подходы к взвешиванию терминов и релевантности:

*   **TF-IDF (1972) / BM25 (1994):** Статистические методы, которые взвешивают термины на основе их частоты в документе (term frequency) и обратной частоты в корпусе (inverse document frequency). Их главное архитектурное отличие – полное отсутствие семантического понимания и контекста; они работают исключительно на поверхностном совпадении токенов.
*   **Query-Document Interaction Models (например, BERT-K (2019) или BERT-based re-rankers):** Эти модели используют BERT для прямого сравнения запроса и документа, вычисляя оценку релевантности. Их архитектурное отличие заключается в том, что они выполняют инференс для *каждой пары* запрос-документ на этапе поиска, что делает их слишком медленными для обработки всего корпуса. DeepCT же стремится *предобработать* документы заранее.
*   **Dense Retrieval Models (например, DPR (2020)):** Модели, которые кодируют запросы и документы в плотные векторы (dense embeddings) и используют векторное расстояние для поиска. Они появились примерно в то же время или чуть позже, чем DeepCT. Их архитектурное отличие – это полная замена инвертированного индекса на поиск по ближайшим соседям в векторном пространстве, тогда как DeepCT *улучшает* существующий инвертированный индекс.

### Архитектура
Архитектура DeepCT относительно проста и состоит из двух основных компонентов:

1.  **Contextualized Encoder:** Это предварительно обученная языковая модель, такая как BERT. Документ разбивается на токены, и каждый токен подается в BERT. BERT генерирует **контекстуализированные эмбеддинги** для каждого токена в документе, учитывая его окружение.
2.  **Term Weighting Head:** Поверх выходных эмбеддингов BERT для каждого токена устанавливается небольшая нейронная сеть (например, однослойный MLP). Эта сеть принимает эмбеддинг токена и предсказывает его **DeepCT score** — числовое значение, отражающее важность этого токена в данном контексте документа.

**Важно:** Для многословных терминов или токенов, которые были разбиты на подслова (subwords) BERT-ом, их DeepCT scores могут быть агрегированы (например, усреднением или взятием максимума) для получения единого веса для исходного слова.

### Алгоритм обучения
Обучение DeepCT является ключевым аспектом, поскольку для контекстуального взвешивания терминов нет прямой разметки. Авторы предлагают метод **слабого надзора (weak supervision)**:

1.  **Создание обучающих пар:** Берется большой набор запросов и документов.
2.  **Генерация "сильных" меток релевантности:** Используется более мощная, но медленная модель (например, обученный BERT-ранкер) для ранжирования документов по отношению к каждому запросу. Документы, которые BERT-ранкер оценивает как высокорелевантные, считаются позитивными примерами.
3.  **Идентификация важных терминов (псевдо-разметка):** Для каждой пары (запрос, высокорелевантный документ) определяется, какие термины в документе внесли наибольший вклад в высокую оценку релевантности от BERT-ранкера. Это можно сделать различными методами, например, анализируя градиенты или внимания (attention weights) BERT-ранкера, или просто по совпадению с запросом. Чем больше термин "помогает" документу быть релевантным, тем выше его целевой DeepCT score.
4.  **Обучение DeepCT модели:** DeepCT модель обучается предсказывать эти "целевые" DeepCT scores для каждого токена в документах.
    *   **Вход:** Документ, токены которого проходят через Contextualized Encoder (BERT).
    *   **Выход:** Для каждого токена предсказывается его DeepCT score.
    *   **Функция потерь:** Используется функция потерь, которая минимизирует расхождение между предсказанными DeepCT scores и целевыми scores (псевдо-метками). Это может быть MSE (Mean Squared Error) или специально разработанная функция потерь, поощряющая правильное ранжирование терминов по важности.

**Ключевая особенность:** DeepCT обучается один раз на большом наборе данных, а затем его веса фиксируются и используются для предобработки всего корпуса.

### Алгоритм инференса
После обучения DeepCT используется для построения улучшенного инвертированного индекса:

1.  **Обработка корпуса:** Для каждого документа в корпусе:
    *   Документ подается на вход обученной DeepCT модели.
    *   Для каждого токена в документе вычисляется его **DeepCT score**.
2.  **Построение/обновление индекса:**
    *   Эти DeepCT scores используются в качестве весов терминов в инвертированном индексе. Вместо традиционной частоты термина (TF) или BM25, индекс хранит DeepCT score для каждого вхождения термина в документе.
    *   Пример: Если слово "apple" появляется в документе "D1" с DeepCT score 0.8, и в документе "D2" с DeepCT score 0.3 (из-за разного контекста), то именно эти значения 0.8 и 0.3 будут использоваться для ранжирования.
3.  **Поиск запроса:**
    *   Когда поступает новый запрос, он обрабатывается традиционным sparse retrieval методом (например, BM25 или простой векторной моделью), но *вместо стандартных TF-IDF/BM25 весов* для документов используются **DeepCT scores**.
    *   Документы ранжируются на основе этих новых, семантически обогащенных весов.
    *   (Опционально) Полученный список документов может быть передан на дальнейшую обработку более точной re-ranker моделью.

### Результаты
DeepCT показал значительные улучшения в эффективности retrieval по сравнению с базовыми Sparse Retrieval моделями:

*   **На датасете MS MARCO:** DeepCT показал улучшение до **~10-20% в MRR@10** (Mean Reciprocal Rank at 10) по сравнению с BM25, что указывает на то, что документы, содержащие релевантные термины в правильном контексте, оказывались выше в списке выдачи.
*   **На TREC Web Track:** На более крупных и разнообразных наборах данных, DeepCT также демонстрировал значимые приросты метрик ранжирования (например, **nDCG@10**), подтверждая свою способность улучшать качество поиска за счет более умного взвешивания терминов.
*   Важно отметить, что DeepCT эффективно работает как **дополнение к существующим sparse retrieval системам**, значительно повышая их качество без кардинального изменения архитектуры или добавления высокой вычислительной стоимости во время инференса запроса (так как веса уже предычислены). Это позволяет "модернизировать" устаревшие методы поиска с помощью последних достижений в LLM.

## 📝 Критический анализ

# DeepCT (2019)
---
[[paper]](https://arxiv.org/pdf/1910.10683)<br>DeepCT = Deep Contextualized Term Weighting

DeepCT — это подход к **контекстуализированному взвешиванию терминов** в документах, использующий языковые модели, такие как BERT, для создания более семантически насыщенных представлений документов в системах информационного поиска. Разработан Microsoft Research.

### Контекст
Традиционные методы, такие как TF-IDF и BM25, основываются на частоте терминов, что не всегда улавливает семантические нюансы. Нейронные модели, такие как BERT, понимают контекст, но их применение для индексирования больших корпусов затруднено из-за вычислительной стоимости.

### Идея
DeepCT использует возможности языковых моделей для обогащения инвертированных индексов, оценивая *важность* каждого слова в контексте документа. Это делает sparse retrieval более семантически осведомленным.

### Постановка задачи
DeepCT улучшает представление документов для **первого этапа поиска**, обеспечивая более точное ранжирование по релевантности.

### Альтернативы
- **TF-IDF / BM25:** Статистические методы без семантического понимания.
- **Query-Document Interaction Models:** Используют BERT для сравнения запроса и документа, но медленны для больших корпусов.
- **Dense Retrieval Models:** Кодируют запросы и документы в плотные векторы, заменяя инвертированный индекс.

### Архитектура
1. **Contextualized Encoder:** Использует BERT для генерации **контекстуализированных эмбеддингов** токенов.
2. **Term Weighting Head:** Нейронная сеть предсказывает **DeepCT score** для каждого токена.

<img src="img/img.png" width=500>

### Обучение
Используется **слабый надзор**:
1. **Создание обучающих пар:** Используются запросы и документы.
2. **Генерация меток релевантности:** Используется BERT-ранкер для оценки релевантности.
3. **Идентификация важных терминов:** Определяются термины, вносящие вклад в релевантность.
4. **Обучение модели:** DeepCT обучается предсказывать целевые DeepCT scores.

### Инференс
1. **Обработка корпуса:** Для каждого документа вычисляются **DeepCT scores**.
2. **Построение индекса:** Используются DeepCT scores вместо традиционных весов.
3. **Поиск запроса:** Документы ранжируются на основе новых весов.

### Результаты
DeepCT улучшает эффективность retrieval:
- **MS MARCO:** Улучшение до **~10-20% в MRR@10** по сравнению с BM25.
- **TREC Web Track:** Значимые приросты метрик ранжирования, такие как **nDCG@10**.

DeepCT эффективно дополняет существующие системы, улучшая качество поиска без значительных изменений архитектуры.

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
import torch.nn as nn
import torch.nn.functional as F

# Step 1: Define the DeepCT model
class DeepCT(nn.Module):
    def __init__(self, bert_model_name='bert-base-uncased'):
        super(DeepCT, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        # Term Weighting Head: a simple linear layer
        self.term_weighting_head = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        # Get contextualized embeddings from BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Use the last hidden state
        last_hidden_state = outputs.last_hidden_state
        # Predict DeepCT scores for each token
        term_scores = self.term_weighting_head(last_hidden_state)
        # Apply a sigmoid to get scores between 0 and 1
        term_scores = torch.sigmoid(term_scores).squeeze(-1)
        return term_scores

# Step 2: Tokenize a sample document
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
document = "DeepCT uses BERT to assign contextualized weights to terms in a document."
inputs = tokenizer(document, return_tensors='pt', truncation=True, padding=True)

# Step 3: Initialize the DeepCT model
deepct_model = DeepCT()

# Step 4: Forward pass to get term weights
with torch.no_grad():
    term_scores = deepct_model(inputs['input_ids'], inputs['attention_mask'])

# Step 5: Display the term weights
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
for token, score in zip(tokens, term_scores[0]):
    print(f"Token: {token}, DeepCT Score: {score.item():.4f}")

# Explanation:
# - We define a DeepCT model that uses BERT to generate contextualized embeddings for each token.
# - A simple linear layer is used as the Term Weighting Head to predict the importance of each token.
# - We tokenize a sample document and pass it through the DeepCT model to get term weights.
# - The term weights represent the importance of each token in the context of the document, as predicted by the model.
```

This code snippet demonstrates the core concept of DeepCT: using a pre-trained BERT model to generate contextualized embeddings for each token in a document, and then predicting a weight for each token using a simple linear layer. The predicted weights (DeepCT scores) reflect the importance of each token in the context of the document, which can be used to enhance traditional inverted indices for information retrieval.